# Sprint 2 Gemma-2 9B - reproducible Kaggle launcher

This launcher clones one exact public Git commit, reads ClearML credentials from Kaggle Secrets without displaying them, requires the official competition data and official attached Gemma-2-9B-IT model, then executes the managed smoke notebook. Use T4 x2 and Internet On. Keep this launcher private while it can access secrets.

In [ ]:
from pathlib import Path

REPOSITORY_URL = 'https://github.com/kujifined/PMLDL-llm-classification-finetuning.git'
EXPECTED_COMMIT = 'cca05136f1ea015e57f1670b875670ee09ee0cca'
EXPERIMENT_ID = 'E20260913115603636696'
MANAGED_NOTEBOOK = f'output/jupyter-notebook/{EXPERIMENT_ID}__sprint-2-gemma-2-9b-lora.ipynb'
REPOSITORY_DIR = Path('/kaggle/working/PMLDL-llm-classification-finetuning')
EXPORT_PATH = Path('/kaggle/working/gemma2-run-output.zip')

In [ ]:
import os
import shutil
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
for secret_name in ('CLEARML_API_ACCESS_KEY', 'CLEARML_API_SECRET_KEY'):
    try:
        secret_value = secret_client.get_secret(secret_name)
    except Exception as exc:
        raise RuntimeError(f'Missing or unavailable Kaggle Secret: {secret_name}') from exc
    if not secret_value:
        raise RuntimeError(f'Kaggle Secret is empty: {secret_name}')
    os.environ[secret_name] = secret_value
del secret_value
os.environ.setdefault('CLEARML_API_HOST', 'https://api.clear.ml')
os.environ.setdefault('CLEARML_WEB_HOST', 'https://app.clear.ml')
os.environ.setdefault('CLEARML_FILES_HOST', 'https://files.clear.ml')

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=REPOSITORY_DIR, check=True)
actual_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=REPOSITORY_DIR,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout.strip()
if actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(f'Git revision mismatch: {actual_commit}')
print('Checked out verified commit:', actual_commit)

In [ ]:
pip_command = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '-r',
    str(REPOSITORY_DIR / 'requirements-baseline.lock'),
    '-r',
    str(REPOSITORY_DIR / 'requirements-transformer.lock'),
    '--no-build-isolation',
    '-e',
    str(REPOSITORY_DIR),
]
subprocess.run(pip_command, check=True)

import_preflight = '''
import torch
import torchvision
import torchaudio
import bitsandbytes
import peft
from transformers import Gemma2ForSequenceClassification
assert torch.__version__.startswith('2.6.0'), torch.__version__
assert torchvision.__version__.startswith('0.21.0'), torchvision.__version__
assert torchaudio.__version__.startswith('2.6.0'), torchaudio.__version__
assert torch.cuda.is_available(), 'A Kaggle GPU accelerator is required'
assert torch.cuda.device_count() >= 2, 'Select Kaggle T4 x2 for the LoRA comparison'
print('Import and T4 x2 preflight OK:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
'''
subprocess.run([sys.executable, '-c', import_preflight], check=True)

In [ ]:
competition_dirs = sorted(
    {
        path.parent.resolve()
        for path in Path('/kaggle/input').glob('**/train.csv')
        if (path.parent / 'test.csv').is_file()
        and (path.parent / 'sample_submission.csv').is_file()
    }
)
if len(competition_dirs) != 1:
    raise RuntimeError(f'Expected one attached official competition input, found {competition_dirs}')
os.environ['PMLDL_DATA_DIR'] = str(competition_dirs[0])
print('Competition data attached at:', competition_dirs[0])

model_candidates = [
    Path('/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2'),
    Path('/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/1'),
    Path('/kaggle/input/gemma-2/transformers/gemma-2-9b-it/2'),
    Path('/kaggle/input/gemma-2/transformers/gemma-2-9b-it/1'),
]
model_dir = next((path for path in model_candidates if (path / 'config.json').is_file()), None)
if model_dir is None:
    discovered = sorted({path.parent.resolve() for path in Path('/kaggle/input').glob('**/config.json') if 'gemma-2-9b-it' in str(path).lower()})
    if len(discovered) == 1:
        model_dir = discovered[0]
    else:
        raise RuntimeError(
            'Attach exactly one official Kaggle model google/gemma-2/transformers/gemma-2-9b-it. '
            f'Discovered candidates: {discovered}'
        )
os.environ['PMLDL_MODEL_DIR'] = str(model_dir)
print('Official Gemma model attached at:', model_dir)

In [ ]:
import tempfile

import nbformat
from nbclient import NotebookClient

source_path = REPOSITORY_DIR / MANAGED_NOTEBOOK
if not source_path.is_file():
    raise FileNotFoundError(source_path)
notebook = nbformat.read(source_path, as_version=4)
client = NotebookClient(
    notebook,
    timeout=None,
    kernel_name='python3',
    resources={'metadata': {'path': str(REPOSITORY_DIR)}},
)
executed_path = Path('/kaggle/working/executed-gemma2-experiment.ipynb')
execution_error = None
try:
    client.execute()
except Exception as exc:
    execution_error = exc
finally:
    nbformat.write(notebook, executed_path)
    with tempfile.TemporaryDirectory(prefix='gemma2-export-', dir='/kaggle/working') as temporary:
        export_root = Path(temporary) / 'gemma2-run-output'
        (export_root / 'results/runs').mkdir(parents=True)
        (export_root / 'artifacts').mkdir(parents=True)
        for run_dir in sorted((REPOSITORY_DIR / 'results/runs').glob(f'{EXPERIMENT_ID}__*')):
            shutil.copytree(run_dir, export_root / 'results/runs' / run_dir.name)
        for artifact_dir in sorted((REPOSITORY_DIR / 'artifacts').glob(f'{EXPERIMENT_ID}__*')):
            shutil.copytree(artifact_dir, export_root / 'artifacts' / artifact_dir.name)
        shutil.copy2(executed_path, export_root / executed_path.name)
        shutil.copy2(
            REPOSITORY_DIR / 'configs/experiments' / f'{EXPERIMENT_ID}.json',
            export_root / f'{EXPERIMENT_ID}.json',
        )
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()
        shutil.make_archive(str(EXPORT_PATH.with_suffix('')), 'zip', export_root)
print('Saved portable run output:', EXPORT_PATH)
if execution_error is not None:
    raise execution_error